[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/13_svm_and_kernels/02_soft_margin_hinge_loss_from_scratch.ipynb)

# Bài 02 — Soft margin, hinge loss và Linear SVM from scratch

**🎯 Mục tiêu**
- Soft-margin SVM: biến **slack** $\xi_i$ cho phép vi phạm lề; tham số $C$ đánh đổi lề rộng ↔ ít vi phạm.
- Viết lại thành bài toán **không ràng buộc** với **hinge loss**: $\frac12\lVert\mathbf w\rVert^2 + C\sum_i\max(0, 1 - y_i(\mathbf w^\top\mathbf x_i + b))$.
- Cài đặt Linear SVM bằng **subgradient descent** (và SGD kiểu Pegasos); đối chiếu với `LinearSVC`.
- So sánh hinge loss với log-loss (Logistic Regression); tác động của $C$.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/13_svm_and_kernels

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC

from ml_course.datasets import make_classification_2d
from ml_course.visualization import PALETTE, plot_decision_boundary, set_style

set_style()

## 1. Khi dữ liệu không tách được: soft margin

Thêm biến slack $\xi_i \ge 0$ — mức độ điểm $i$ vi phạm lề:

$$\min_{\mathbf w, b, \xi}\ \frac12\lVert\mathbf w\rVert^2 + C\sum_i \xi_i \quad\text{s.t.}\quad y_i(\mathbf w^\top\mathbf x_i + b) \ge 1 - \xi_i,\ \ \xi_i \ge 0$$

- $\xi_i = 0$: đúng phía và ngoài lề; $0 < \xi_i \le 1$: đúng phía nhưng **trong lề**; $\xi_i > 1$: **sai phía**.
- $C$ lớn → phạt vi phạm nặng → lề hẹp, bám dữ liệu (variance cao); $C$ nhỏ → lề rộng, chấp nhận vi phạm (bias cao).

Tại nghiệm tối ưu, $\xi_i = \max(0, 1 - y_i(\mathbf w^\top\mathbf x_i + b))$ → bài toán tương đương **không ràng buộc**:

$$\min_{\mathbf w, b}\ \frac12\lVert\mathbf w\rVert^2 + C\sum_i \underbrace{\max\big(0,\ 1 - y_i(\mathbf w^\top\mathbf x_i + b)\big)}_{\text{hinge loss}}$$

Giống hệt dạng "loss + regularization" quen thuộc (chương 06–07), chỉ khác hàm loss!

In [ ]:
m = np.linspace(-2, 3, 300)
plt.plot(m, np.maximum(0, 1 - m), label="hinge: max(0, 1 − m)")
plt.plot(m, np.log2(1 + np.exp(-m)), label="log-loss (logistic) / ln 2")
plt.plot(m, (m < 0).astype(float), label="0-1 loss")
plt.axvline(1, color="gray", ls=":")
plt.xlabel("margin m = y·(wᵀx + b)")
plt.ylabel("loss")
plt.legend()
plt.title("Hinge loss = 0 khi m ≥ 1: điểm 'đủ xa' hoàn toàn không ảnh hưởng → support vectors")
plt.show()

Điểm khác biệt then chốt: hinge loss **bằng 0** khi margin ≥ 1 → các điểm đó không đóng góp gradient → nghiệm chỉ phụ thuộc vào **support vectors** (điểm có $m \le 1$). Log-loss luôn > 0 → mọi điểm đều ảnh hưởng.

## 2. Subgradient descent

Hinge loss không khả vi tại $m = 1$, nhưng có **subgradient**:

$$\frac{\partial}{\partial\mathbf w} = \mathbf w - C\sum_{i:\ m_i < 1} y_i\mathbf x_i, \qquad \frac{\partial}{\partial b} = -C\sum_{i:\ m_i < 1} y_i$$

In [ ]:
class LinearSVM:
    def __init__(self, C=1.0, lr=0.01, n_iters=3000):
        self.C, self.lr, self.n_iters = C, lr, n_iters

    def fit(self, X, y):
        ys = np.where(y == 1, 1.0, -1.0)
        n, d = X.shape
        self.w, self.b = np.zeros(d), 0.0
        self.history = []
        for t in range(self.n_iters):
            margins = ys * (X @ self.w + self.b)
            viol = margins < 1                                            # điểm vi phạm lề (support vectors "hiện tại")
            self.history.append(0.5 * self.w @ self.w + self.C * np.maximum(0, 1 - margins).sum())
            grad_w = self.w - self.C * (ys[viol, None] * X[viol]).sum(0)
            grad_b = -self.C * ys[viol].sum()
            step = self.lr / np.sqrt(1 + t)                               # bước giảm dần (cần cho subgradient)
            self.w -= step * grad_w
            self.b -= step * grad_b
        return self

    def decision_function(self, X):
        return X @ self.w + self.b

    def predict(self, X):
        return (self.decision_function(X) >= 0).astype(int)


X, y = make_classification_2d("blobs", n=400, noise=0.6, seed=5)
X = StandardScaler().fit_transform(X)
ours = LinearSVM(C=1.0, lr=0.005, n_iters=5000).fit(X, y)
sk = LinearSVC(C=1.0, loss="hinge", max_iter=100_000, tol=1e-6).fit(X, y)


def primal_objective(w, b, C=1.0):
    ys = np.where(y == 1, 1.0, -1.0)
    return 0.5 * w @ w + C * np.maximum(0, 1 - ys * (X @ w + b)).sum()


print(f"scratch: w = {ours.w.round(3)}, b = {ours.b:.3f} | objective = {primal_objective(ours.w, ours.b):.4f}")
print(f"sklearn: w = {sk.coef_[0].round(3)}, b = {sk.intercept_[0]:.3f} | objective = {primal_objective(sk.coef_[0], sk.intercept_[0]):.4f}")
plt.plot(ours.history)
plt.yscale("log")
plt.xlabel("vòng lặp")
plt.ylabel("objective")
plt.title("Subgradient descent: objective giảm (không đơn điệu tuyệt đối)")
plt.show()

## 3. Pegasos: SGD cho SVM với dữ liệu lớn

Shalev-Shwartz et al. (2007): mỗi bước chọn **một** mẫu ngẫu nhiên, bước $\eta_t = \frac{1}{\lambda t}$ (với $\lambda = \frac{1}{Cn}$). Hội tụ nhanh, bộ nhớ $O(d)$ — nền tảng cho SVM tuyến tính trên hàng triệu mẫu (`SGDClassifier(loss="hinge")`).

In [ ]:
def pegasos(X, y, lam=0.01, n_epochs=20, seed=0):
    r = np.random.default_rng(seed)
    ys = np.where(y == 1, 1.0, -1.0)
    Xb = np.c_[X, np.ones(len(X))]                 # gộp bias vào w (bias cũng bị regularize — biến thể đơn giản)
    w = np.zeros(Xb.shape[1])
    t = 0
    for _ in range(n_epochs):
        for i in r.permutation(len(X)):
            t += 1
            eta = 1 / (lam * t)
            if ys[i] * (Xb[i] @ w) < 1:
                w = (1 - eta * lam) * w + eta * ys[i] * Xb[i]
            else:
                w = (1 - eta * lam) * w
    return w


w_peg = pegasos(X, y, lam=1 / len(X))
acc = np.mean(((np.c_[X, np.ones(len(X))] @ w_peg) >= 0).astype(int) == y)
print(f"Pegasos: w = {w_peg[:2].round(3)}, b = {w_peg[2]:.3f} | accuracy train = {acc:.3f} | LinearSVC: {sk.score(X, y):.3f}")

## 4. Tác động của C

In [ ]:
Xc, yc = make_classification_2d("blobs", n=200, noise=0.9, seed=11)
Xc = StandardScaler().fit_transform(Xc)
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
grid = np.linspace(-3, 3, 50)
for ax, C in zip(axes, [0.01, 1, 100]):
    svc = SVC(kernel="linear", C=C).fit(Xc, yc)
    w, b = svc.coef_[0], svc.intercept_[0]
    for c in [0, 1]:
        ax.scatter(Xc[yc == c, 0], Xc[yc == c, 1], color=PALETTE[c], s=20)
    for offset, style in [(0, "k-"), (1, "k--"), (-1, "k--")]:
        ax.plot(grid, (offset - b - w[0] * grid) / w[1], style, lw=1)
    ax.scatter(*Xc[svc.support_].T, s=90, facecolors="none", edgecolors="k", lw=0.8)
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_title(f"C = {C}: lề rộng 2/‖w‖ = {2 / np.linalg.norm(w):.2f}, {len(svc.support_)} support vectors")
plt.show()

- $C$ nhỏ: lề **rất rộng**, rất nhiều support vectors (mọi điểm trong lề), ranh giới ổn định.
- $C$ lớn: lề hẹp, ít support vectors, ranh giới bị kéo bởi vài điểm khó.

Lưu ý: $C$ trong SVM đóng vai trò **giống** $C$ trong `LogisticRegression` — $C = 1/\lambda$, $C$ lớn = regularization yếu.

## 5. SVM vs Logistic Regression

In [ ]:
Xm, ym = make_classification_2d("linear", n=600, noise=0.4, seed=2)
X_tr, X_te, y_tr, y_te = train_test_split(StandardScaler().fit_transform(Xm), ym, test_size=0.4, random_state=0)
for name, model in [("Linear SVM (hinge)", LinearSVC(C=1.0, max_iter=10_000)), ("Logistic Regression", LogisticRegression(C=1.0))]:
    model.fit(X_tr, y_tr)
    print(f"{name:20s} accuracy test = {model.score(X_te, y_te):.3f} | có predict_proba: {hasattr(model, 'predict_proba')}")

Trên dữ liệu tuyến tính, hai model thường gần như tương đương về accuracy. Khác biệt thực tế:
- LR cho **xác suất** calibrated tự nhiên; SVM chỉ cho **điểm số** (khoảng cách tới siêu phẳng) — cần Platt scaling nếu muốn xác suất (bài 04).
- SVM thưa (support vectors) và có **kernel trick** hiệu quả (bài 03).
- Hinge loss bỏ qua điểm "đủ xa" → ít bị ảnh hưởng bởi các điểm dễ; LR bị ảnh hưởng bởi mọi điểm.

## 🧠 Kiểm tra nhanh

1. Một điểm có $y_i(\mathbf w^\top\mathbf x_i + b) = 0.4$. Slack $\xi_i$ và hinge loss bằng bao nhiêu? Điểm nằm ở đâu?
2. Vì sao soft-margin SVM với ràng buộc tương đương với bài toán hinge loss không ràng buộc?
3. Tăng $C$ thì lề và số support vectors thay đổi thế nào?
4. Vì sao subgradient descent cần learning rate giảm dần?

## 📌 Tóm tắt

- Soft margin: slack $\xi_i$, tham số $C$; tương đương $\frac12\lVert\mathbf w\rVert^2 + C\sum\text{hinge}$.
- Hinge = 0 khi margin ≥ 1 → nghiệm chỉ phụ thuộc support vectors.
- Giải bằng subgradient descent / Pegasos (SGD) / solver chuyên dụng (liblinear).
- $C$ lớn: lề hẹp, variance cao; $C$ nhỏ: lề rộng, bias cao.

## 📚 Tài liệu tham khảo

- Shalev-Shwartz, Singer & Srebro (2007) — *Pegasos: Primal Estimated sub-GrAdient SOlver for SVM*
- ESL §12.2–12.3
- Vũ Hữu Tiệp — [Soft Margin SVM](https://machinelearningcoban.com/2017/04/13/softmarginsmv/)